In [15]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.cluster import KMeans
from sklearn.metrics import (
    precision_recall_curve, auc, cohen_kappa_score, f1_score,
    recall_score, precision_score, accuracy_score, confusion_matrix,
    roc_auc_score, balanced_accuracy_score, matthews_corrcoef,
    log_loss, brier_score_loss, fbeta_score
)
from sklearn.calibration import CalibratedClassifierCV
from sklearn.isotonic import IsotonicRegression
from sklearn.preprocessing import MinMaxScaler
import time
from pathlib import Path
import logging
import os
import sys
import traceback
import importlib.util
from typing import Dict, Tuple, List, Any, Optional
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from google.colab import drive
import warnings

# --- Logging Configuration ---
# Filter out the specific PFL warning
logging.getLogger('PFLZ_module_temp').setLevel(logging.ERROR)
logging.getLogger('PFLZ_module_temp').propagate = False
logging.getLogger('PFLZ').setLevel(logging.ERROR)
logging.getLogger('PFLZ').propagate = False

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    handlers=[
        logging.StreamHandler(),
        logging.FileHandler('/tmp/pflz_lightgbm.log')
    ]
)
logger = logging.getLogger(__name__)

# Definition of the PFL_CP class as a simple placeholder/interface.
# Its methods will be provided by the dynamically loaded PFL module.

class PFLZ_CP:
    def __init__(self, n_clusters: int, epsilon: float = 1e-8, lgb_params: Optional[Dict[str, Any]] = None, feature_names: Optional[List[str]] = None) -> None:
        self.n_clusters = n_clusters
        self.epsilon = epsilon
        self.lgb_params = lgb_params if lgb_params is not None else {}
        self.feature_names = feature_names
        self.cluster_centers_ = np.array([])
        self.cluster_model = None
        self.cond_probs_ = None
        self.n_original_features_ = 0
        self.n_fuzzy_features_ = n_clusters
        self.X_original_fit = None

# Global variable to track if the actual PFL module has been successfully loaded
_PFLZ_MODULE_LOADED = False

# --- Module 1: Centralized Configuration ---

class SystemConfiguration:
    """Centralized configuration with validation and fixed clinical thresholds."""

    NB_FEATURES = None
    NB_RULES = 7
    INDICES_FEATURES = None
    EPSILON = 1e-8

    CLINICAL_RISK_THRESHOLDS = {
        'LOW': 10.0,
        'MEDIUM': 15.0, # Medium risk is between 10% and 15%.
        'HIGH': 25.0 # High risk starts from 25%.
    }

    LGB_PARAMS = {
        'objective': 'binary',
        'metric': 'binary_logloss',
        'boosting_type': 'gbdt',
        'learning_rate': 0.08,
        'num_leaves': 25,
        'max_depth': 4,
        'min_data_in_leaf': 10,
        'feature_fraction': 0.8,
        'bagging_fraction': 0.85,
        'bagging_freq': 5,
        'lambda_l1': 0.3,
        'lambda_l2': 0.6,
        'min_gain_to_split': 0.02,
        'path_smooth': 100,
        'verbose': -1,
        'random_state': 42,
        'force_col_wise': True,
        'scale_pos_weight': 1
    }

    COHERENCE_LAMBDA = 0.15
    COHERENCE_ALERT_THRESHOLD = 0.10

    PERCENTILE_COVERAGE = 60

    CALIBRATION_METHOD = 'isotonic'
    CALIBRATION_CV = 3

    SCALERS = {
        'min_max_transformed': {},
        'raw_stats': {},
        'feature_scaler_obj': None,
        'feature_scaler_features': []
    }


    @classmethod
    def validate_configuration(cls):
        """Validates configuration consistency."""
        if cls.NB_FEATURES is not None and cls.INDICES_FEATURES is not None:
            if len(cls.INDICES_FEATURES) != cls.NB_FEATURES:
                raise ValueError("Inconsistency between NB_FEATURES and INDICES_FEATURES")
        logger.info("✓ Configuration validated")

# --- Module 2: Data Management ---

class DataManager:
    """Manages Drive mounting, locating, and loading data."""
    def __init__(self):
        self.mydrive_path = None
        self.modeling_data2_path = None
        self.all_feature_names = None
        self.df_train = None
        self.X_train_df = None
        self.y_train_arr = None
        self.X_train_raw = None


    def mount_drive(self):
        """Mounts the Drive into the file system."""
        print("="*50)
        print("MOUNTING DRIVE")
        print("="*50)
        drive.mount('/content/drive', force_remount=True)
        self.mydrive_path = Path("/content/drive/MyDrive")
        print("✓ Drive mounted successfully")

    def locate_data_folder(self):
        """Locates the 'modeling_data2' folder in the Drive."""
        print("\n" + "="*50)
        print("LOCATING DATA FOLDER")
        print("="*50)
        pfl_dir_candidate = self.mydrive_path / "PFL"
        if pfl_dir_candidate.is_dir() and (pfl_dir_candidate / "modeling_data2").is_dir():
            self.modeling_data2_path = pfl_dir_candidate / "modeling_data2"
            print(f"✓ modeling_data2 folder found: {self.modeling_data2_path}")
            return

        for root, dirs, _ in os.walk(self.mydrive_path):
            if "modeling_data2" in dirs:
                self.modeling_data2_path = Path(root) / "modeling_data2"
                print(f"✓ modeling_data2 folder found: {self.modeling_data2_path}")
                return
        raise FileNotFoundError("modeling_data2 folder not found in Drive")

    def load_data(self):
        """Loads training data (df_train, X_train, y_train) from the folder."""
        print("\n" + "="*50)
        print("LOADING DATA")
        print("="*50)
        df_train_path = self.modeling_data2_path / 'df_train.hdf'
        if not df_train_path.exists():
            raise FileNotFoundError(f"File not found: {df_train_path}")
        self.df_train = pd.read_hdf(df_train_path, key='Processed_data')

        self.X_train_raw = pd.read_hdf(self.modeling_data2_path / 'X_train.hdf', key='Processed_data').copy()

        X_train_df_temp = pd.read_hdf(self.modeling_data2_path / 'X_train.hdf', key='Processed_data')
        self.all_feature_names = [col for col in X_train_df_temp.columns if col != 'A']

        A_train = self.df_train['A'].values
        self.X_train_df = pd.read_hdf(self.modeling_data2_path / 'X_train.hdf', key='Processed_data')
        self.X_train_df['A'] = A_train

        self.y_train_arr = pd.read_hdf(self.modeling_data2_path / 'y_train.hdf', key='Processed_data').values.ravel()

        transformed_min_max_values = {}
        for feature_name in self.all_feature_names:
            if feature_name in self.X_train_df.columns and pd.api.types.is_numeric_dtype(self.X_train_df[feature_name]):
                min_val = self.X_train_df[feature_name].min()
                max_val = self.X_train_df[feature_name].max()
                transformed_min_max_values[feature_name] = (min_val, max_val)
        SystemConfiguration.SCALERS['min_max_transformed'] = transformed_min_max_values


        # --- Calculate and store statistics of ORIGINAL RAW data (for denormalization and interpretation) ---
        # NOTE: The prints in this section have been moved to _collect_hypothetical_patient_inputs
        raw_stats = {}
        features_for_scaler = [col for col in self.X_train_raw.columns if col != 'A']

        X_for_scaler = self.X_train_raw[features_for_scaler].select_dtypes(include=np.number)

        if X_for_scaler.isnull().any().any():
            X_for_scaler = X_for_scaler.fillna(X_for_scaler.mean())
            logger.warning("NaNs were found in the raw data and imputed with the mean for the MinMaxScaler.")

        scaler = MinMaxScaler()
        scaler.fit(X_for_scaler)
        SystemConfiguration.SCALERS['feature_scaler_obj'] = scaler
        SystemConfiguration.SCALERS['feature_scaler_features'] = X_for_scaler.columns.tolist()


        for feature_name in features_for_scaler:
            if feature_name in self.X_train_raw.columns and pd.api.types.is_numeric_dtype(self.X_train_raw[feature_name]):
                col_data = self.X_train_raw[feature_name].dropna()
                if not col_data.empty:
                    raw_stats[feature_name] = {
                        'min': col_data.min(),
                        'max': col_data.max(),
                        'mean': col_data.mean(),
                        '25th_percentile': col_data.quantile(0.25),
                        '50th_percentile': col_data.quantile(0.50),
                        '75th_percentile': col_data.quantile(0.75)
                    }
            else:
                logger.warning(f"Feature '{feature_name}' is not numeric in the raw data for statistics.")
        SystemConfiguration.SCALERS['raw_stats'] = raw_stats

        print("\n✅ Data loaded successfully!")
        return self.X_train_df, self.y_train_arr, self.all_feature_names

# --- Module 3: PFL_CP Module Loader ---

class PFLZModuleLoader:
    """Manages the dynamic loading of the PFL class (alias PFL_CP) from its external file
    ensuring it is correctly imported."""
    @staticmethod
    def load_pflz_cp_module(mydrive_path: Path):
        print("\n" + "="*50)
        print("LOADING PFL_CP MODULE")
        print("="*50)
        pflz_file_path = None
        candidate_path = mydrive_path / "PFL" / "Modules" / "PFLZ.py"
        if candidate_path.exists():
            pflz_file_path = candidate_path
        else:
            for root, dirs, files in os.walk(mydrive_path):
                if "PFL" in dirs:
                    pfl_dir = Path(root) / "PFL"
                    if "Modules" in os.listdir(pfl_dir):
                        modules_path = pfl_dir / "Modules"
                        if "PFLZ.py" in os.listdir(modules_path):
                            pflz_file_path = modules_path / "PFLZ.py"
                            break

        if pflz_file_path is None:
            raise FileNotFoundError("PFL.py file not found in Drive. Please ensure PFL.py exists in the 'PFL/Modules' folder.")

        print(f"✓ PFL.py file found: {pflz_file_path}")

        spec = importlib.util.spec_from_file_location("PFLZ_module_temp", pflz_file_path)
        if spec is None:
            raise ImportError(f"Could not specify module for {pflz_file_path}.")

        module = importlib.util.module_from_spec(spec)
        sys.modules["PFLZ_module_temp"] = module
        spec.loader.exec_module(module)

        loaded_pflz_class = getattr(module, 'PFLZ', None)

        if loaded_pflz_class is None:
            raise ImportError("The class 'PFL' could not be found in the loaded module. Check the class name in PFLZ.py.")

        class PFLZ_CP_Alias(loaded_pflz_class):
            def __init__(self, *args, **kwargs):
                super().__init__(*args, **kwargs)
                if not hasattr(self, 'X_original_fit'):
                    self.X_original_fit = None

        return PFLZ_CP_Alias

# --- Module 4: Interpretability Calculations ---

class InterpretabilityCalculator:
    """Calculates interpretability scores and generates clinical rules from the PFL model."""

    @staticmethod
    def calculate_interpretability_features(pflz_cp, X: np.ndarray, y: np.ndarray) -> Tuple[np.ndarray, Dict, Dict]:
        """
        Calculates interpretability features (scores, confidence, dominant clusters)
        and cluster interpretations (fuzzy rules).
        """
        logger.info("Calculating interpretability features...")

        try:
            membership_values = pflz_cp.calc_mf(X, normalized=True)
            if membership_values is None or membership_values.size == 0 or membership_values.shape[1] == 0:
                raise ValueError("Invalid or empty fuzzy membership values after calc_mf.")
        except Exception as e:
            logger.error(f"Error calculating membership values: {e}")
            raise

        try:
            conditional_probs = pflz_cp.calc_cond_probs(X, y)
            if conditional_probs is None or conditional_probs.size == 0 or conditional_probs.shape[1] < 2:
                raise ValueError("Invalid or empty conditional probabilities after calc_cond_probs.")
        except Exception as e:
            logger.error(f"Error calculating conditional probabilities: {e}")
            raise

        n_samples = X.shape[0]
        scores_bruts = np.zeros(n_samples)
        dominant_clusters = np.zeros(n_samples, dtype=int)
        confidence_scores = np.zeros(n_samples)

        num_clusters_actual = min(pflz_cp.n_clusters, membership_values.shape[1])

        for i in range(n_samples):
            membership_i = membership_values[i, :num_clusters_actual]
            weighted_probs = membership_i * conditional_probs[:num_clusters_actual, 1]

            if weighted_probs.size > 0:
                max_weighted_prob = np.max(weighted_probs)
                scores_bruts[i] = max_weighted_prob
                dominant_clusters[i] = np.argmax(weighted_probs)
            else:
                scores_bruts[i] = 0.0
                dominant_clusters[i] = -1

            confidence_scores[i] = np.max(membership_i) if membership_i.size > 0 else 0.0

        if np.max(scores_bruts) > np.min(scores_bruts):
            scores_normalises = (
                (scores_bruts - np.min(scores_bruts)) /
                (np.max(scores_bruts) - np.min(scores_bruts))
            )
        else:
            scores_normalises = scores_bruts

        adaptive_threshold = np.percentile(scores_normalises,
                                             SystemConfiguration.PERCENTILE_COVERAGE)

        diagnostics = InterpretabilityCalculator._diagnostiquer_distributions(
            scores_bruts,
            scores_normalises,
            adaptive_threshold
        )

        interpretability_features = np.column_stack([
            scores_normalises,
            confidence_scores,
            dominant_clusters
        ])

        cluster_interpretations = {}
        if hasattr(pflz_cp, 'cluster_centers_') and pflz_cp.cluster_centers_.shape[0] > 0:
            rule_widths = pflz_cp.calc_rule_widths()
        else:
            logger.warning("PFL_CP centroids are not available for rule width calculation. Using default widths.")
            num_features = X.shape[1] if X.shape[0] > 0 else 1
            rule_widths = np.full((pflz_cp.n_clusters, num_features), 0.5)

        alpha_rule = 1.5
        for cluster_id in range(num_clusters_actual):
            if hasattr(pflz_cp, 'cluster_centers_') and pflz_cp.cluster_centers_.shape[0] > cluster_id and pflz_cp.feature_names is not None:
                cluster_center = pflz_cp.cluster_centers_[cluster_id]
                denormalized_rule = InterpretabilityCalculator._generer_regle_centroide(
                    cluster_center,
                    pflz_cp.feature_names,
                    SystemConfiguration.SCALERS.get('min_max_transformed', {}), # This parameter is no longer used for final denormalization but is kept for the signature
                    alpha=alpha_rule,
                    widths=rule_widths[cluster_id] if rule_widths.shape[0] > cluster_id else np.full(cluster_center.shape[0], 0.5)
                )
                cluster_interpretations[cluster_id] = {
                    'prob_conditionnelle': conditional_probs[cluster_id, 1],
                    'regle': denormalized_rule
                }
            else:
                logger.warning(f"Cluster {cluster_id} or feature names are not available. Skipping rule generation for this cluster.")

        logger.info(f"✓ Interpretability features calculated - Adaptive threshold: {adaptive_threshold:.3f}")

        return interpretability_features, cluster_interpretations, diagnostics

    @staticmethod
    def _diagnostiquer_distributions(scores_bruts: np.ndarray, scores_normalises: np.ndarray,
                                   adaptive_threshold: float) -> Dict:
        """Provides a detailed diagnosis of interpretability score distributions."""
        diagnostics = {
            'scores_bruts_stats': {
                'min': np.min(scores_bruts), 'max': np.max(scores_bruts),
                'mean': np.mean(scores_bruts), 'std': np.std(scores_bruts),
                'percentiles': np.percentile(scores_bruts, [25, 50, 75, 90, 95])
            },
            'scores_normalises_stats': {
                'min': np.min(scores_normalises), 'max': np.max(scores_normalises),
                'mean': np.mean(scores_normalises), 'std': np.std(scores_normalises),
                'percentiles': np.percentile(scores_normalises, [25, 50, 75, 90, 95])
            },
            'seuil_adaptatif': adaptive_threshold,
            'coverage_adaptatif': np.mean(scores_normalises > adaptive_threshold)
        }

        diagnostics['scores_interpretabilite_normalises_original'] = scores_normalises
        logger.info(f"Diagnostic - Adaptive coverage: {diagnostics['coverage_adaptatif']:.1%}")
        return diagnostics

    @staticmethod
    def _generer_regle_centroide(centroid: np.ndarray, feature_names: List[str], scalers_transformed: Dict = None, alpha: float = 1.0, widths: Optional[np.ndarray] = None) -> str:
        """
        Generates a readable clinical rule from a centroid, with denormalization.
        Uses `feature_names` directly (which are the relevant features).
        Alpha is used to adjust the interval width.
        `widths` provides specific widths for each centroid feature.
        """
        conditions = []
        num_features_to_process = min(len(centroid), len(feature_names))
        if len(centroid) != len(feature_names):
            logger.warning(f"Inconsistency: Centroid has {len(centroid)} elements but {len(feature_names)} feature names. Processing the first {num_features_to_process}...")

        if widths is None or len(widths) != num_features_to_process:
            logger.warning(f"Widths not provided or incorrect dimension for _generer_regle_centroide. Using a default width.")
            default_width = 0.5
            widths = np.full(num_features_to_process, default_width)

        scaler_obj = SystemConfiguration.SCALERS.get('feature_scaler_obj')
        raw_stats = SystemConfiguration.SCALERS.get('raw_stats', {})
        scaler_features_order = SystemConfiguration.SCALERS.get('feature_scaler_features', [])

        if scaler_obj is None or not raw_stats or not scaler_features_order:
            logger.error("Scaler or raw statistics are missing for rule denormalization. Rules will not be denormalized correctly.")
            for i in range(num_features_to_process):
                original_feature_name = feature_names[i]
                normalized_value = centroid[i]
                feature_width = widths[i]
                lower_bound_norm = normalized_value - alpha * feature_width
                upper_bound_norm = normalized_value + alpha * feature_width
                conditions.append(f"{original_feature_name} ∈ [{lower_bound_norm:.3f}, {upper_bound_norm:.3f}] (Internal Normalized)")
        else:
            centroid_for_inverse_transform = np.zeros(len(scaler_features_order))
            feature_name_to_original_idx = {name: i for i, name in enumerate(feature_names)}

            for i, scaler_feature_name in enumerate(scaler_features_order):
                if scaler_feature_name in feature_name_to_original_idx:
                    centroid_for_inverse_transform[i] = centroid[feature_name_to_original_idx[scaler_feature_name]]
                else:
                    centroid_for_inverse_transform[i] = 0.5

            denormalized_center = scaler_obj.inverse_transform(centroid_for_inverse_transform.reshape(1, -1))[0]

            for i, original_feature_name in enumerate(feature_names):
                if original_feature_name not in raw_stats:
                    logger.warning(f"Raw statistics are missing for '{original_feature_name}'. Denormalization and interpretation are limited.")
                    normalized_value = centroid[i]
                    feature_width = widths[i]
                    lower_bound_norm = normalized_value - alpha * feature_width
                    upper_bound_norm = normalized_value + alpha * feature_width
                    conditions.append(f"{original_feature_name} ∈ [{lower_bound_norm:.3f}, {upper_bound_norm:.3f}] (Internal Normalized)")
                    continue

                try:
                    scaler_idx = scaler_features_order.index(original_feature_name)
                    denormalized_val = denormalized_center[scaler_idx]
                except ValueError:
                    logger.warning(f"Feature '{original_feature_name}' not found in scaler features order. Skipping denormalization for this feature.")
                    normalized_value = centroid[i]
                    feature_width = widths[i]
                    lower_bound_norm = normalized_value - alpha * feature_width
                    upper_bound_norm = normalized_value + alpha * feature_width
                    conditions.append(f"{original_feature_name} ∈ [{lower_bound_norm:.3f}, {upper_bound_norm:.3f}] (Internal Normalized)")
                    continue

                raw_min = raw_stats[original_feature_name]['min']
                raw_max = raw_stats[original_feature_name]['max']
                raw_range = raw_max - raw_min

                denormalized_interval_half_width = alpha * widths[i] * raw_range

                lower_bound_denorm = denormalized_val - denormalized_interval_half_width
                upper_bound_denorm = denormalized_val + denormalized_interval_half_width

                p25 = raw_stats[original_feature_name]['25th_percentile']
                p50 = raw_stats[original_feature_name]['50th_percentile']
                p75 = raw_stats[original_feature_name]['75th_percentile']

                position_interpretation = ""
                center_interval = (lower_bound_denorm + upper_bound_denorm) / 2

                if center_interval < p25:
                    position_interpretation = "rather low"
                elif center_interval > p75:
                    position_interpretation = "rather high"
                else:
                    position_interpretation = "average"

                rule_range = upper_bound_denorm - lower_bound_denorm

                if raw_range > 1e-8:
                    ratio_width = rule_range / raw_range
                    if ratio_width < 0.10:
                        width_interpretation = "very precise"
                    elif ratio_width < 0.30:
                        width_interpretation = "quite precise"
                    else:
                        width_interpretation = "wide"
                else:
                    width_interpretation = "constant"

                conditions.append(f"{original_feature_name} ∈ [{lower_bound_denorm:.2f}, {upper_bound_denorm:.2f}] (value {position_interpretation}, range {width_interpretation})")

        simplified_rule = " AND ".join(conditions)

        return simplified_rule

# --- Module 5: Advanced Metrics with Calibration ---

class MetricsCalculator:
    """Calculates advanced performance metrics and manages post-training calibration."""

    @staticmethod
    def calculate_ece(y_true: np.ndarray, y_pred_proba: np.ndarray, n_bins: int = 10) -> float:
        """Calculates Expected Calibration Error (ECE)."""
        bin_edges = np.linspace(0, 1, n_bins + 1)
        bin_indices = np.digitize(y_pred_proba, bin_edges[:-1])
        ece = 0.0
        for i in range(1, n_bins + 1):
            in_bin = bin_indices == i
            bin_count = np.sum(in_bin)
            if bin_count > 0:
                bin_true_mean = np.mean(y_true[in_bin])
                bin_pred_mean = np.mean(y_pred_proba[in_bin])
                ece += np.abs(bin_true_mean - bin_pred_mean) * (bin_count / len(y_true))
        return ece

    @staticmethod
    def apply_post_training_calibration(y_true: np.ndarray, y_pred_proba: np.ndarray,
                                         method: str = 'isotonic') -> Tuple[np.ndarray, object]:
        """Applies post-training calibration to improve the reliability of probabilities."""
        logger.info(f"Applying {method} calibration...")
        if method == 'isotonic':
            calibrator = IsotonicRegression(out_of_bounds='clip')
        elif method == 'sigmoid':
            calibrator = LogisticRegression(solver='lbfgs')
        else:
            raise ValueError(f"Unknown calibration method: {method}")

        if method == 'sigmoid':
            logits = np.log(np.clip(y_pred_proba, 1e-15, 1-1e-15) /
                            np.clip(1-y_pred_proba, 1e-15, 1-1e-15))
            calibrator.fit(logits.reshape(-1, 1), y_true)
            y_pred_calibrated = calibrator.predict_proba(logits.reshape(-1, 1))[:, 1]
        else:
            calibrator.fit(y_pred_proba, y_true)
            y_pred_calibrated = calibrator.predict(y_pred_proba)

        logger.info("✓ Calibration applied")
        return y_pred_calibrated, calibrator

    @staticmethod
    def _print_metric_definitions():
        """Prints definitions of performance metrics."""
        print("\n📊 METRIC DEFINITIONS:")
        print("    • y: Observed outcome - 1 (72h Mortality), 0 (Survival)")
        print("    • A: Antibiotic treatment - 1 (Single), 0 (Multiple)")

# --- Module 6: Training with Calibration and Constraints ---

class ModelTrainer:
    """Trains the tree-based model with calibration and integrates interpretability features."""

    def __init__(self, config: SystemConfiguration):
        self.config = config
        self.config.validate_configuration()
        self.calibrators = {}

    def train_and_evaluate(self, X_subset: pd.DataFrame, y_subset: np.ndarray,
                                   subset_name: str, selected_features: List[str]) -> Dict:
        """Trains and evaluates the tree-based model for a given subset."""
        logger.info(f"Starting processing for {subset_name} with the tree-based model.")
        start_time = time.time()

        global _PFLZ_MODULE_LOADED
        if not _PFLZ_MODULE_LOADED:
            raise RuntimeError("The PFL module was not loaded correctly. Cannot continue training.")

        try:
            pflz_cp = PFLZ_CP(
                n_clusters=self.config.NB_RULES,
                epsilon=self.config.EPSILON,
                lgb_params=self.config.LGB_PARAMS,
                feature_names=selected_features
            )
            pflz_cp.fit(X_subset.values, y_subset)
            pflz_cp.X_original_fit = X_subset.values

        except Exception as e:
            logger.error(f"Error during PFL_CP initialization or fitting for {subset_name}: {e}")
            raise

        centroids = pflz_cp.cluster_centers_

        interp_features, cluster_explanations, diagnostics = (
            InterpretabilityCalculator.calculate_interpretability_features(
                pflz_cp, X_subset.values, y_subset
            )
        )

        self._display_diagnostics(diagnostics, subset_name)
        self._display_cluster_explanations(cluster_explanations)

        results_train_eval, trained_lgbm_model = self._cross_validation(
            X_subset.values, y_subset,
            pflz_cp, interp_features, subset_name, diagnostics
        )

        self._display_results(results_train_eval, subset_name)

        results_train_eval['time_minutes'] = round((time.time() - start_time) / 60, 2)
        logger.info(f"✓ Processing for {subset_name} completed in {results_train_eval['time_minutes']} min for the tree-based model.")

        return {
            'centroids': centroids,
            'predictions': results_train_eval['y_pred_proba_calibre'],
            'raw_predictions': results_train_eval['y_pred_proba'],
            'results': results_train_eval,
            'pflz_cp': pflz_cp,
            'explanations': cluster_explanations,
            'diagnostics': diagnostics,
            'calibrator': self.calibrators.get(subset_name),
            'trained_lgbm_model': trained_lgbm_model
        }

    def _cross_validation(self, X: np.ndarray, y: np.ndarray, pflz_cp: Any,
                                   interp_features: np.ndarray, subset_name: str,
                                   diagnostics: Dict) -> Tuple[Dict, Any]:
        """Performs cross-validation of the tree-based model with calibration and feature weighting."""
        logger.info(f"Starting cross-validation for the tree-based model ({subset_name}).")

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

        metrics = {
            'accuracy': [], 'balanced_accuracy': [], 'kappa': [], 'mcc': [],
            'precision': [], 'recall': [], 'f1_score': [],
            'specificity': [], 'pr_auc': [], 'roc_auc': [], 'log_loss': [], 'brier': [],
            'fdr': [], 'for': []
        }

        y_pred_proba_full = np.zeros(len(y))
        y_pred_full = np.zeros(len(y))
        coherence_gaps_fold = []
        trained_model = None

        unique_classes, counts = np.unique(y, return_counts=True)
        if len(unique_classes) > 1:
            majority_class_count = counts[0] if unique_classes[0] == 0 else counts[1]
            minority_class_count = counts[1] if unique_classes[1] == 1 else counts[0]
            scale_pos_weight_val = majority_class_count / minority_class_count if minority_class_count > 0 else 1
        else:
            scale_pos_weight_val = 1

        for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
            logger.debug(f"Processing fold {fold+1}/5 for the tree-based model.")

            X_train_fold = X[train_idx]
            X_val_fold = X[val_idx]
            y_train_fold = y[train_idx]
            y_val_fold = y[val_idx]

            fuzzy_train = pflz_cp.calc_mf(X_train_fold, normalized=True)
            fuzzy_val = pflz_cp.calc_mf(X_val_fold, normalized=True)

            interp_train = interp_features[train_idx]
            interp_val = interp_features[val_idx]

            interpretability_weight = 15.0

            weighted_interp_train = interp_train * interpretability_weight
            weighted_interp_val = interp_val * interpretability_weight

            if fuzzy_train.shape[1] == 0 and self.config.NB_RULES > 0:
                fuzzy_train = np.zeros((X_train_fold.shape[0], self.config.NB_RULES))
            if fuzzy_val.shape[1] == 0 and self.config.NB_RULES > 0:
                fuzzy_val = np.zeros((X_val_fold.shape[0], self.config.NB_RULES))

            if weighted_interp_train.shape[1] == 0:
                weighted_interp_train = np.zeros((X_train_fold.shape[0], 3))
            if weighted_interp_val.shape[1] == 0:
                weighted_interp_val = np.zeros((X_val_fold.shape[0], 3))

            X_train_extended = np.hstack((X_train_fold, fuzzy_train, weighted_interp_train))
            X_val_extended = np.hstack((X_val_fold, fuzzy_val, weighted_interp_val))

            lgb_params_fold = self.config.LGB_PARAMS.copy()
            lgb_params_fold['scale_pos_weight'] = scale_pos_weight_val

            train_data = lgb.Dataset(X_train_extended, label=y_train_fold, free_raw_data=False)
            val_data = lgb.Dataset(X_val_extended, label=y_val_fold, reference=train_data, free_raw_data=False)

            model = lgb.train(
                lgb_params_fold,
                train_data,
                num_boost_round=500,
                valid_sets=[val_data],
                callbacks=[
                    lgb.early_stopping(stopping_rounds=30, verbose=False),
                ]
            )
            trained_model = model

            y_pred_proba = model.predict(X_val_extended)
            y_pred = (y_pred_proba > 0.2).astype(int)

            y_pred_proba_full[val_idx] = y_pred_proba
            y_pred_full[val_idx] = y_pred

            if interp_val.shape[1] > 0:
                ecart_fold = np.mean(np.abs(y_pred_proba - interp_val[:, 0]))
            else:
                ecart_fold = 1.0
            coherence_gaps_fold.append(ecart_fold)

            self._calculate_metriques_fold(y_val_fold, y_pred, y_pred_proba, metrics)

        y_pred_proba_calibrated, calibrator = MetricsCalculator.apply_post_training_calibration(
            y, y_pred_proba_full, method=self.config.CALIBRATION_METHOD
        )
        self.calibrators[subset_name] = calibrator

        ece_raw = MetricsCalculator.calculate_ece(y, y_pred_proba_full)
        ece_calibrated = MetricsCalculator.calculate_ece(y, y_pred_proba_calibrated)

        return ({
            'metriques': metrics,
            'y_pred_proba': y_pred_proba_full,
            'y_pred_proba_calibre': y_pred_proba_calibrated,
            'y_pred': y_pred_full,
            'ece_brut': ece_raw,
            'ece_calibre': ece_calibrated,
            'amelioration_ece': ece_raw - ece_calibrated,
            'ecarts_coherence_folds': coherence_gaps_fold,
            'ecart_coherence_moyen': np.mean(coherence_gaps_fold)
        }, trained_model)

    def _calculate_metriques_fold(self, y_true: np.ndarray, y_pred: np.ndarray, y_pred_proba: np.ndarray, metrics: Dict):
        """Calculates performance metrics for a single fold, handling single-class cases."""
        try:
            y_pred_proba = np.clip(y_pred_proba, 1e-15, 1 - 1e-15)

            if len(np.unique(y_true)) == 1 or len(np.unique(y_pred)) == 1:
                tn, fp, fn, tp = (0,0,0,0)
                if len(np.unique(y_true)) == 1 and y_true[0] == 0:
                    tn = np.sum(y_pred == 0)
                    fp = np.sum(y_pred == 1)
                elif len(np.unique(y_true)) == 1 and y_true[0] == 1:
                    tp = np.sum(y_pred == 1)
                    fn = np.sum(y_pred == 0)
                else:
                    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
                    tn, fp, fn, tp = cm.ravel()
            else:
                cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
                tn, fp, fn, tp = cm.ravel()

            specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
            fdr = fp / (tp + fp) if (tp + fp) > 0 else 0
            for_rate = fn / (tn + fn) if (tn + fn) > 0 else 0

            metrics['accuracy'].append(accuracy_score(y_true, y_pred))
            metrics['balanced_accuracy'].append(balanced_accuracy_score(y_true, y_pred))
            metrics['kappa'].append(cohen_kappa_score(y_true, y_pred))
            metrics['mcc'].append(matthews_corrcoef(y_true, y_pred))
            metrics['precision'].append(precision_score(y_true, y_pred, zero_division=0))
            metrics['recall'].append(recall_score(y_true, y_pred, zero_division=0))
            metrics['f1_score'].append(f1_score(y_true, y_pred, zero_division=0))
            metrics['specificity'].append(specificity)

            if len(np.unique(y_true)) > 1:
                precision_vals, recall_vals, _ = precision_recall_curve(y_true, y_pred_proba)
                metrics['pr_auc'].append(auc(recall_vals, precision_vals))
                metrics['roc_auc'].append(roc_auc_score(y_true, y_pred_proba))
            else:
                metrics['pr_auc'].append(np.nan)
                metrics['roc_auc'].append(np.nan)

            metrics['log_loss'].append(log_loss(y_true, y_pred_proba))
            metrics['brier'].append(brier_score_loss(y_true, y_pred_proba))
            metrics['fdr'].append(fdr)
            metrics['for'].append(for_rate)

        except Exception as e:
            logger.error(f"Error calculating metrics for a fold: {e}. Filling with 0.0 or NaN.")
            for key in metrics.keys():
                if key in ['pr_auc', 'roc_auc']:
                    metrics[key].append(np.nan)
                else:
                    metrics[key].append(0.0)

    def _display_diagnostics(self, diagnostics: Dict, subset_name: str):
        """Displays interpretability diagnostics."""
        print(f"\n📊 INTERPRETABILITY DIAGNOSTICS FOR {subset_name}")
        print("-" * 60)
        print(f"Raw Scores - Min: {diagnostics['scores_bruts_stats']['min']:.4f}, "
              f"Max: {diagnostics['scores_bruts_stats']['max']:.4f}, "
              f"Mean: {diagnostics['scores_bruts_stats']['mean']:.4f}")
        print(f"Normalized Scores - Min: {diagnostics['scores_normalises_stats']['min']:.4f}, "
              f"Max: {diagnostics['scores_normalises_stats']['max']:.4f}, "
              f"Mean: {diagnostics['scores_normalises_stats']['mean']:.4f}")
        print(f"Adaptive Threshold ({SystemConfiguration.PERCENTILE_COVERAGE}th percentile): {diagnostics['seuil_adaptatif']:.4f}")
        print(f"Adaptive Coverage: {diagnostics['coverage_adaptatif']:.1%}")

    def _display_cluster_explanations(self, cluster_explanations: Dict):
        """Displays cluster interpretations (associated fuzzy rules)."""
        print("\n📋 Cluster-based Interpretations:")
        for cluster_id, explanation in cluster_explanations.items():
            print(f"\nCluster {cluster_id}:")
            print(f"    • P(mortality|cluster) = {explanation['prob_conditionnelle']:.2%}")
            print(f"    • Rule: {explanation['regle']}")

    def _display_results(self, results: Dict, subset_name: str):
        """Displays performance and calibration metrics of the tree-based model."""
        print(f"\n{'-'*70}")
        print(f"METRICS FOR {subset_name} (tree-based model)")
        print(f"{'-'*70}")

        MetricsCalculator._print_metric_definitions()

        mean_metrics = {
            'Accuracy': np.nanmean(results['metriques']['accuracy']) * 100,
            'Balanced Accuracy': np.nanmean(results['metriques']['balanced_accuracy']) * 100,
            'Kappa': np.nanmean(results['metriques']['kappa']),
            'MCC': np.nanmean(results['metriques']['mcc']),
            'Precision': np.nanmean(results['metriques']['precision']) * 100,
            'Recall': np.nanmean(results['metriques']['recall']) * 100,
            'F1-Score': np.nanmean(results['metriques']['f1_score']) * 100,
            'Specificity': np.nanmean(results['metriques']['specificity']) * 100,
            'PR-AUC': np.nanmean(results['metriques']['pr_auc']),
            'ROC-AUC': np.nanmean(results['metriques']['roc_auc']),
            'Log Loss': np.nanmean(results['metriques']['log_loss']),
            'Brier': np.nanmean(results['metriques'].get('brier', [np.nan])),
            'FDR': np.nanmean(results['metriques']['fdr']),
            'FOR': np.nanmean(results['metriques']['for'])
        }

        calibration_metrics = {
            'Raw ECE': results.get('ece_brut', np.nan),
            'Calibrated ECE': results.get('ece_calibre', np.nan),
            'ECE Improvement': results.get('amelioration_ece', np.nan),
        }

        print("\n📈 PERFORMANCE RESULTS of the tree-based model:")
        print("Accuracy                                  : {:.2f}%".format(mean_metrics['Accuracy']))
        print("Balanced Accuracy                         : {:.2f}%".format(mean_metrics['Balanced Accuracy']))
        print("Kappa                                     : {:.4f}".format(mean_metrics['Kappa']))
        print("MCC                                       : {:.4f}".format(mean_metrics['MCC']))
        print("Precision                                 : {:.2f}%".format(mean_metrics['Precision']))
        print("Recall                                    : {:.2f}%".format(mean_metrics['Recall']))
        print("F1-Score                                  : {:.2f}%".format(mean_metrics['F1-Score']))
        print("Specificity                               : {:.2f}%".format(mean_metrics['Specificity']))
        print("PR-AUC                                    : {:.4f}".format(mean_metrics['PR-AUC'] if not np.isnan(mean_metrics['PR-AUC']) else 0.0))
        print("ROC-AUC                                   : {:.4f}".format(mean_metrics['ROC-AUC'] if not np.isnan(mean_metrics['ROC-AUC']) else 0.0))
        print("Log Loss                                  : {:.4f}".format(mean_metrics['Log Loss']))
        print("Brier Score                               : {:.4f}".format(mean_metrics['Brier'] if not np.isnan(mean_metrics['Brier']) else 0.0))
        print("FDR                                       : {:.4f}".format(mean_metrics['FDR']))
        print("FOR                                       : {:.4f}".format(mean_metrics['FOR']))

        print("\n🎯 CALIBRATION METRICS of the tree-based model:")
        for name, value in calibration_metrics.items():
            if 'ECE' in name:
                print(f"    {name:<20}: {value:.4f}")
            else:
                print(f"    {name:<20}: {value:.4f}")

        print(f"{'-'*70}")

# --- Module 7: Risk Cluster Analyzer ---

class RiskClusterAnalyzer:
    """Analyzes risk clusters with fixed clinical thresholds and coherence alerts."""

    @staticmethod
    def analyze_risk_clusters(X: pd.DataFrame, centroids: np.ndarray,
                                     predictions: np.ndarray, subset_name: str,
                                     selected_features: List[str],
                                     pflz_cp: Any, cluster_explanations: Dict,
                                     raw_predictions: Optional[np.ndarray] = None,
                                     y_true: Optional[np.ndarray] = None,
                                     normalized_interp_scores: Optional[np.ndarray] = None):
        """
        Performs an analysis by cluster, classifying risk based on fixed clinical thresholds
        and evaluating coherence between the tree-based model and fuzzy rules.
        """
        logger.info(f"Analyzing risk clusters for {subset_name}")

        if centroids.shape[0] == 0 or X.empty or len(selected_features) == 0:
            logger.warning(f"Centroids, X data, or selected features are empty for risk cluster analysis of {subset_name}. Skipping detailed cluster analysis.")
            return {'coherence_alerts': []}

        valid_features_for_clustering = [f for f in selected_features if f in X.columns]
        if not valid_features_for_clustering:
            logger.warning(f"None of the selected features ({selected_features}) are present in the X DataFrame. Cannot perform clustering for {subset_name}. Skipping.")
            return {'coherence_alerts': []}

        km = KMeans(n_clusters=len(centroids), init=centroids, n_init=1, random_state=42)
        clusters = km.fit_predict(X[valid_features_for_clustering].values)

        print(f"\n{'-'*70}")
        print(f"RISK CLUSTERS FOR {subset_name}")
        print(f"{'-'*70}")

        thresholds = SystemConfiguration.CLINICAL_RISK_THRESHOLDS
        print("🏥 FIXED CLINICAL THRESHOLDS:")
        print(f"- LOW RISK: < {thresholds['LOW']:.1f}%")
        print(f"- MEDIUM RISK: {thresholds['LOW']:.1f}% to {thresholds['MEDIUM']:.1f}%")
        print(f"- HIGH RISK: ≥ {thresholds['HIGH']:.1f}%")
        print(f"{'-'*70}")

        global_mean_risk = np.mean(predictions) * 100
        print(f"📊 Global mean risk of the subset: {global_mean_risk:.1f}%")

        cluster_coherence_alerts = []

        for i in range(len(centroids)):
            cluster_mask = (clusters == i)
            n_patients = np.sum(cluster_mask)

            if n_patients == 0:
                print(f"\nCluster {i}: No patients")
                continue

            if i not in cluster_explanations:
                logger.warning(f"Cluster ID {i} was not found in 'cluster_explanations'. Skipping this cluster for detailed analysis.")
                continue

            tree_model_mean_risk = np.mean(predictions[cluster_mask]) * 100

            if raw_predictions is not None:
                tree_model_raw_risk = np.mean(raw_predictions[cluster_mask]) * 100
                calibration_improvement = abs(tree_model_raw_risk - tree_model_mean_risk)
            else:
                tree_model_raw_risk = tree_model_mean_risk
                calibration_improvement = 0

            if tree_model_mean_risk < thresholds['LOW']:
                risk_level = "LOW"
                probable_outcome = "Probable Survival"
                emoji_color = "🟢"
            elif tree_model_mean_risk <= thresholds['MEDIUM']:
                risk_level = "MEDIUM"
                probable_outcome = "Enhanced Surveillance"
                emoji_color = "🟡"
            else:
                risk_level = "HIGH"
                probable_outcome = "Urgent Intervention"
                emoji_color = "🔴"

            conditional_prob = cluster_explanations[i].get('prob_conditionnelle', 0.0)
            simplified_rule = cluster_explanations[i].get('regle', 'Rule not available')

            ecart_coherence_cluster = abs(tree_model_mean_risk/100 - conditional_prob)

            specific_coherence_alert = ecart_coherence_cluster > SystemConfiguration.COHERENCE_ALERT_THRESHOLD
            if specific_coherence_alert:
                cluster_coherence_alerts.append({
                    'cluster': i,
                    'gap': ecart_coherence_cluster,
                    'tree_model_risk': tree_model_mean_risk,
                    'rules_prob': conditional_prob * 100
                })

            cluster_fidelity = 1 - ecart_coherence_cluster
            cluster_fidelity = max(0, min(1, cluster_fidelity))

            print(f"\n{emoji_color} Cluster {i} - {risk_level.upper()} RISK")
            print(f"    • Calibrated risk (Tree-based model): {tree_model_mean_risk:.1f}%")
            if raw_predictions is not None:
                print(f"    • Raw risk (Tree-based model): {tree_model_raw_risk:.1f}% (improvement: {calibration_improvement:.1f}%)")
            print(f"    • P(mortality|cluster) (Fuzzy Rule): {conditional_prob:.2%}")
            print(f"    • Cluster Fidelity                                   : {cluster_fidelity:.4f}")
            print(f"    • Clinical Recommendation: {probable_outcome}")
            print(f"    • Number of patients: {n_patients:,}")
            print(f"    • Percentage of subset: {(n_patients/len(predictions))*100:.1f}%")

            print(f"    • Clinical rule: {simplified_rule}")

            coherence_status_display = ""
            if ecart_coherence_cluster < (SystemConfiguration.COHERENCE_ALERT_THRESHOLD / 2):
                coherence_status_display = "🟢 Excellent"
            elif ecart_coherence_cluster < SystemConfiguration.COHERENCE_ALERT_THRESHOLD:
                coherence_status_display = "🟡 Good"
            else:
                coherence_status_display = "🔴 Poor - In-depth analysis required"

            print(f"    • Rule-tree model coherence: {coherence_status_display} (gap: {ecart_coherence_cluster:.1%})")

            if specific_coherence_alert:
                print(f"    ⚠️ ALERT: Significant gap detected - For this cluster, consider:")
                print("      - A thorough review of feature interactions.")
                print("      - Validation with clinical expertise for these patient profiles.")

            print(f"{'-'*70}")

        if cluster_coherence_alerts:
            print(f"\n🚨 SUMMARY OF CLUSTER COHERENCE ALERTS ({len(cluster_coherence_alerts)} cluster(s)):")
            for alert in cluster_coherence_alerts:
                print(f"    • Cluster {alert['cluster']}: tree-based model={alert['tree_model_risk']:.1f}% vs Rules={alert['rules_prob']:.1f}% (Gap: {alert['gap']:.1%})")
            print("\n💡 General Recommendations for Discrepancies:")
            print("    - **Rule Review:** Collaborate with clinical expertise to refine or create more precise rules for these patient profiles.")
            print("    - **Feature Enrichment:** Consider adding additional features to the tree-based model.")
        else:
            print(f"\n✅ SATISFACTORY CLUSTER COHERENCE: No significant gaps detected.")


        logger.info(f"✓ Fidelity and coherence analysis completed for {subset_name}")
        return {
            'coherence_alerts': cluster_coherence_alerts
        }

# --- Module 8: Fidelity & Coherence Analyzer ---

class FidelityCoherenceAnalyzer:
    """Calculates and presents global fidelity and coherence metrics of the model."""
    def __init__(self, config: SystemConfiguration):
        self.config = config

    def analyze_interpretability_measures(self, tree_model_pred_proba: np.ndarray,
                                            interp_feature_scores: np.ndarray,
                                            y_true: np.ndarray,
                                            pflz_cp_instance: Any,
                                            subset_name: str,
                                            cluster_explanations_details: Dict):
        """
        Calculates and displays global fidelity and coherence metrics,
        as well as calibration diagnostics.
        """
        logger.info(f"Analyzing fidelity and coherence measures for {subset_name}...")

        normalized_interp_scores = interp_feature_scores[:, 0]

        global_mean_gap = np.mean(np.abs(tree_model_pred_proba - normalized_interp_scores))
        global_coherence_calc = 1 - global_mean_gap

        global_fidelity_calc = global_coherence_calc

        adaptive_threshold = np.percentile(normalized_interp_scores, self.config.PERCENTILE_COVERAGE)
        adaptive_coverage_calc = np.mean(normalized_interp_scores >= adaptive_threshold) * 100

        print(f"\n{'-'*70}")
        print(f"FIDELITY & COHERENCE SUMMARY FOR {subset_name}")
        print(f"{'-'*70}")
        print(f"📊 Global Coherence (1 - Mean Absolute Gap): {global_coherence_calc:.4f}")
        print(f"📊 Global Mean Absolute Gap: {global_mean_gap:.4f}")
        print(f"📊 Global Fidelity: {global_fidelity_calc:.4f}")
        print(f"📊 Adaptive Coverage ({self.config.PERCENTILE_COVERAGE}th percentile): {adaptive_coverage_calc:.2f}%")

        if global_mean_gap > self.config.COHERENCE_ALERT_THRESHOLD:
            print("🚨 GLOBAL COHERENCE ALERT: The gap between model predictions and rule explanations is significant.")
            print("    A more in-depth analysis of cluster-level discrepancies is recommended.")
        else:
            print("✅ SATISFACTORY GLOBAL COHERENCE: Model predictions are well-aligned with rule interpretations.")

        X_df_for_plotting = pd.DataFrame()
        if hasattr(pflz_cp_instance, 'X_original_fit') and pflz_cp_instance.X_original_fit is not None and pflz_cp_instance.feature_names is not None:
            if pflz_cp_instance.X_original_fit.shape[1] == len(pflz_cp_instance.feature_names):
                X_df_for_plotting = pd.DataFrame(pflz_cp_instance.X_original_fit, columns=pflz_cp_instance.feature_names)
            else:
                logger.warning(f"Dimension mismatch between X_original_fit ({pflz_cp_instance.X_original_fit.shape[1]} cols) and feature_names ({len(pflz_cp_instance.feature_names)} names). Skipping DataFrame creation for cluster analysis.")
        else:
            logger.warning("Original X data or feature names are not available in PFL_CP instance. Skipping cluster visualizations.")

        logger.info(f"✓ Fidelity and coherence analysis completed for {nom_subset}")
        return {
            'coherence_alerts': []
        }

# --- Main Pipeline ---

class PipelinePrincipal:
    """Main pipeline for clinical integration, orchestrating all steps."""

    def __init__(self):
        self.config = SystemConfiguration()
        self.data_manager = DataManager()
        self.X_train = None
        self.y_train = None
        self.all_feature_names = None
        self.features_selectionnees = None

        self.config.validate_configuration()

        logger.info("🗝️ Main pipeline initialized.")

    def _initialiser_et_charger_donnees(self):
        """Data initialization: mounts Drive, locates and loads datasets."""
        self.data_manager.mount_drive()
        self.data_manager.locate_data_folder()
        self.X_train, self.y_train, self.all_feature_names = self.data_manager.load_data()

        if self.X_train is None or self.y_train is None:
            raise ValueError("Data loading failed")

        self.features_selectionnees = [col for col in self.X_train.columns if col != 'A']
        SystemConfiguration.NB_FEATURES = len(self.features_selectionnees)
        SystemConfiguration.INDICES_FEATURES = list(range(SystemConfiguration.NB_FEATURES))

        print("\n" + "="*50)
        print("SYSTEM CONFIGURATION")
        print("="*50)
        print("🏥 FIXED CLINICAL THRESHOLDS:")
        for level, threshold in self.config.CLINICAL_RISK_THRESHOLDS.items():
            print(f"    • {level.upper()} RISK: {threshold}%")
        print("\n🔧 Functions:")
        print("    • ✅ Adaptive thresholds for coverage (60th percentile)")
        print("    • ✅ Normalization of interpretability scores (0-1)")
        print("    • ✅ Post-training calibration (isotonic)")
        print("    • ✅ Alert system for inconsistencies (threshold: 15%)")
        print("    • ✅ Increased weighting of interpretability features (boosted to 15)")
        print("    • ✅ Fixed clinical thresholds for comparability")
        print("    • ✅ Detailed logging and diagnostics")
        print("    • ✅ Class weighting to improve recall")
        print("    • ✅ Classification threshold adjusted to 0.2 to improve recall")
        print("    • ✅ Number of PFL rules: 7")
        print("    • ✅ All features (except 'A') used for rules")
        print("    • ✅ Rule interval widths adjusted (alpha=1.5)")

        print("="*50)

    def _charger_module_pflz(self):
        """Loads the PFL module from the specified directory."""
        global PFLZ_CP
        global _PFLZ_MODULE_LOADED
        try:
            loaded_pflz_class = PFLZModuleLoader.load_pflz_cp_module(self.data_manager.mydrive_path)
            if loaded_pflz_class is None:
                raise ImportError("The PFL class could not be found or loaded.")
            PFLZ_CP = loaded_pflz_class
            _PFLZ_MODULE_LOADED = True
            logger.info("✓ PFL_CP module loaded and validated")
        except Exception as e:
            logger.error(f"❌ Error loading PFL_CP: {e}")
            _PFLZ_MODULE_LOADED = False
            raise


    def _preparer_subsets(self):
        """Prepares data subsets based on the 'A' variable."""
        logger.info("Preparing data subsets.")
        mask_A0 = self.X_train['A'] == 0
        mask_A1 = self.X_train['A'] == 1

        X_train_no_A = self.X_train.drop('A', axis=1)

        X_train_A0 = X_train_no_A[mask_A0]
        y_train_A0 = self.y_train[mask_A0]
        X_train_A1 = X_train_no_A[mask_A1]
        y_train_A1 = self.y_train[mask_A1]
        X_train_all = X_train_no_A
        y_train_all = self.y_train

        logger.info(f"Dataset A=0: {len(X_train_A0)} patients ({np.mean(y_train_A0):.1%} mortality)")
        logger.info(f"Dataset A=1: {len(X_train_A1)} patients ({np.mean(y_train_A1):.1%} mortality)")
        logger.info(f"Dataset ALL: {len(X_train_all)} patients ({np.mean(y_train_all):.1%} mortality)")
        return {
            'A0': (X_train_A0, y_train_A0, "A=0"),
            'A1': (X_train_A1, y_train_A1, "A=1"),
            'ALL': (X_train_all, y_train_all, "A=Tous")
        }

    def _afficher_resume_final(self, results: Dict, cluster_analyses: Dict):
        """Displays a final summary of system results."""
        print("\n" + "="*60)
        print("FINAL SUMMARY - PFL-TREE-BASED SYSTEM")
        print("="*60)

        print("\n🎯 Functions:")
        print("    • ✅ Adaptive thresholds for Coverage (60th percentile)")
        print("    • ✅ Normalization of interpretability scores (0-1)")
        print("    • ✅ Post-training calibration (isotonic)")
        print("    • ✅ Alert system for inconsistencies (threshold: 15%)")
        print("    • ✅ Increased weighting of interpretability features for the tree-based model (15)")
        print("    • ✅ Denormalized rules for increased clinical readability (alpha=1.5)")
        print("    • ✅ Detailed logging and diagnostics")
        print("    • ✅ Class weighting to improve recall")
        print("    • ✅ Classification threshold adjusted to 0.2 to improve recall")
        print("    • ✅ Number of PFL rules: 7")
        print("    • ✅ All features (except 'A') used for rules")

        print("\n📊 RESULTS PER DATASET:")
        for subset_name, data in results.items():
            results = data['results']
            specific_analysis_results = cluster_analyses.get(subset_name, {}).get('alertes_coherence', [])

            print(f"\n📈 {subset_name}:")
            print(f"    • ROC-AUC of the tree-based model: {np.nanmean(results['metriques']['roc_auc'])*100:.2f}%")
            print(f"    • Raw ECE: {results.get('ece_brut', np.nan):.4f}")
            print(f"    • Calibrated ECE: {results.get('ece_calibre', np.nan):.4f}")
            print(f"    • ECE Improvement: {results.get('amelioration_ece', np.nan):.4f}")
            print(f"    • Cluster inconsistency alerts: {len(specific_analysis_results)}")
            print(f"    • Recall: {np.nanmean(results['metriques']['recall'])*100:.2f}%")

        print("\n🏥 CLINICAL IMPACT:")
        print("    • Better calibrated probabilities → Increased confidence in tree-based model predictions.")
        print("    • Fixed thresholds → Comparability between patient populations.")
        print("    • Adaptive coverage → More patients with reliable interpretations (detailed by the dedicated analysis).")
        print("    • Cluster-level alert system → Targeted detection of complex cases for the tree-based model.")
        print("    • Detailed diagnostics → Debugging and continuous improvement.")
        print("    • Denormalized rules → Facilitated clinical decision-making.")
        print("    • Improved recall → Better detection of mortality cases.")

        print("\n🎯 RECOMMENDATIONS:")
        print("    1. 🟢 Excellent Coherence (< 5% gap per cluster): Use rules as the main guide, with the tree-based model confirming.")
        print("    2. 🟡 Good Coherence (5-10% gap per cluster): Combine rules and tree-based model with caution, surveillance required.")
        print("    3. 🔴 Poor Coherence (> 10% gap per cluster): Prioritize tree-based model predictions and investigate complex interactions.")
        print("    4. 📊 Regularly monitor the tree-based model's calibration with new patients.")
        print("    5. 🔄 Periodically recalibrate if a degradation in ECE is observed for the tree-based model.")

        print("\n" + "="*60)
        print("✅ SYSTEM READY FOR CLINICAL USE")
        print("="*60)
        logger.info("✅ Final summary displayed successfully.")

    def _predict_and_explain_single_instance(self, pflz_cp_instance: Any, cluster_explanations: Dict, selected_features: List[str], X_train_for_means: pd.DataFrame, user_input_values_collected: Dict):
        """
        Performs the analysis of the hypothetical patient using the trained models
        and the user inputs already collected.
        """
        scaler_obj = self.config.SCALERS.get('feature_scaler_obj')
        scaler_features_order = self.config.SCALERS.get('feature_scaler_features', [])

        if scaler_obj is None or not scaler_features_order:
            logger.error("Feature scaler not found or not trained in the configuration. Cannot proceed with patient analysis.")
            return

        X_hypothetical = pd.DataFrame(columns=selected_features)
        X_train_means_no_A = X_train_for_means.drop('A', errors='ignore').mean()
        default_row = X_train_means_no_A.to_dict()
        X_hypothetical.loc[0] = default_row

        for feature, value in user_input_values_collected.items():
            X_hypothetical.loc[0, feature] = value

        X_hypothetical_ordered_for_transform = X_hypothetical[scaler_features_order]
        X_hypothetical_ordered_filled = X_hypothetical_ordered_for_transform.fillna(X_train_means_no_A[scaler_features_order])

        X_hypothetical_normalized_values = scaler_obj.transform(X_hypothetical_ordered_filled.values)

        fuzzy_memberships = pflz_cp_instance.calc_mf(X_hypothetical_normalized_values, normalized=True)

        if hasattr(pflz_cp_instance, 'cond_probs_') and pflz_cp_instance.cond_probs_ is not None and pflz_cp_instance.cond_probs_.shape[0] > 0:
            num_clusters_actual = min(fuzzy_memberships.shape[1], pflz_cp_instance.cond_probs_.shape[0])
            weighted_interpretability_scores = fuzzy_memberships[0, :num_clusters_actual] * pflz_cp_instance.cond_probs_[:num_clusters_actual, 1]

            if np.sum(weighted_interpretability_scores) == 0:
                dominant_cluster_idx = np.argmax(fuzzy_memberships[0, :num_clusters_actual])
            else:
                dominant_cluster_idx = np.argmax(weighted_interpretability_scores)

            if dominant_cluster_idx in cluster_explanations:
                explanation = cluster_explanations[dominant_cluster_idx]
                rule = explanation['regle']
                cluster_inherent_mortality_prob = explanation['prob_conditionnelle']
            else:
                rule = "Rule not found for this cluster."
                cluster_inherent_mortality_prob = np.nan

            print("\n" + "="*70)
            print("RESULT FOR THE HYPOTHETICAL PATIENT")
            print("="*70)
            print(f"The hypothetical patient has the following characteristics (original values):")
            for feat, val in user_input_values_collected.items():
                print(f" - {feat}: {val:.2f}")
            if len(user_input_values_collected) < len(selected_features):
                unspecified_features = [f for f in selected_features if f not in user_input_values_collected]
                print(f" - Other unspecified features: Using default values (training averages).")

            print(f"\nThis patient belongs most strongly to Cluster {dominant_cluster_idx}.")
            print(f"The rule associated with this cluster indicates:")
            print(f"    • P(mortality|cluster) = {cluster_inherent_mortality_prob:.2%}")
            print(f"    • Detailed rule: {rule}")
            print("\n" + "="*70)
            print("ANALYSIS COMPLETED FOR THE HYPOTHETICAL PATIENT")
            print("="*70)
        else:
            print("\nUnable to generate a detailed interpretation for the hypothetical patient.")
            print("Conditional probabilities for the clusters are not available or valid.")

    def _collect_hypothetical_patient_inputs(self, selected_features: List[str], X_train_for_means: pd.DataFrame) -> Dict:
        """
        Displays features and collects user inputs for the hypothetical patient.
        This method is called early in the interaction flow.
        """
        print("\n" + "="*70)
        print("PREPARATION FOR HYPOTHETICAL PATIENT ANALYSIS")
        print("="*70)

        # This section displays the min/max of the features

        print("\nHere is the list of available features with their real ranges (Min/Max):")
        raw_stats = self.config.SCALERS.get('raw_stats', {})
        for i, feature_name in enumerate(selected_features):
            if feature_name in raw_stats:
                min_val = raw_stats[feature_name]['min']
                max_val = raw_stats[feature_name]['max']
                print(f"  {i+1}. {feature_name}: Real Min = {min_val:.4f}, Real Max = {max_val:.4f}")
            else:
                print(f"  {i+1}. {feature_name}: Real statistics not available (not numeric or outside of stats).")

        print("\n--- Enter values for the hypothetical patient ---")
        num_features_to_spec = -1
        MAX_FEATURES_TO_SPEC = len(selected_features)

        while num_features_to_spec < 0 or num_features_to_spec > MAX_FEATURES_TO_SPEC:
            try:
                num_features_to_spec_str = input(f"How many features do you want to specify (0-{MAX_FEATURES_TO_SPEC})? ").strip()
                num_features_to_spec = int(num_features_to_spec_str)
            except ValueError:
                print("Please enter a valid integer.")

        user_input_values = {}
        if num_features_to_spec > 0:
            chosen_indices = []
            valid_indices_entered = False
            while not valid_indices_entered:
                try:
                    feature_numbers_str = input(f"Enter the NUMBERS with commas between features (e.g., 1, {MAX_FEATURES_TO_SPEC}): ").strip()
                    input_parts = [part.strip() for part in feature_numbers_str.split(',') if part.strip()]

                    temp_indices = []
                    all_parts_valid = True
                    for part in input_parts:
                        if part.isdigit():
                            idx = int(part) - 1
                            if 0 <= idx < len(selected_features):
                                if idx not in temp_indices:
                                    temp_indices.append(idx)
                                else:
                                    print(f"Warning: Feature {int(part)} was entered twice and will be ignored.")
                            else:
                                print(f"Invalid feature number: {int(part)}. Please enter all numbers again.")
                                all_parts_valid = False
                                break
                        else:
                            print(f"Invalid format for '{part}'. Please enter only integers separated by commas.")
                            all_parts_valid = False
                            break

                    if all_parts_valid:
                        if len(temp_indices) != num_features_to_spec:
                            print(f"You asked to specify {num_features_to_spec} features, but you entered {len(temp_indices)}. Please try again.")
                        else:
                            chosen_indices = temp_indices
                            valid_indices_entered = True

                except Exception as e:
                    print(f"Error reading feature numbers: {e}. Please try again.")

            for idx in chosen_indices:
                chosen_feature_name = selected_features[idx]
                valid_value = False
                while not valid_value:
                    try:
                        value = float(input(f"Enter the value for '{chosen_feature_name}': "))
                        if chosen_feature_name in raw_stats:
                            min_v = raw_stats[chosen_feature_name]['min']
                            max_v = raw_stats[chosen_feature_name]['max']
                            if not (min_v <= value <= max_v):
                                print(f"Warning: The value ({value:.2f}) is outside the real range ({min_v:.2f}-{max_v:.2f}) observed in your training data for '{chosen_feature_name}'.")
                        user_input_values[chosen_feature_name] = value
                        valid_value = True
                    except ValueError:
                        print("Please enter a valid numeric value.")

        return user_input_values


# --- Main entry point ---

if __name__ == "__main__":
    try:
        warnings.filterwarnings('ignore') # Ignorer les avertissements pour une sortie plus propre
        logger.info("🚀 Démarrage du système PFL.")
        pipeline = PipelinePrincipal()

        # --- PREMIÈRE INTERACTION UTILISATEUR (avant le chargement) ---
        print("\n" + "="*70)
        print("CHOIX DU GROUPE DE TRAITEMENT ANTIBIOTIQUE")
        print("="*70)
        print("Please choose the antibiotic treatment group for analysis:")
        print(" - '0' for patients on multiple antibiotics")
        print(" - '1' for patients on a single antibiotic")
        print(" - 'ALL' for all patients")

        chosen_subset_key = ""
        valid_choices = ['0', '1', 'ALL']
        while chosen_subset_key not in valid_choices:
            chosen_subset_key = input("Please choose the antibiotic treatment group for analysis (0, 1, or ALL) : ").strip().upper()
            if chosen_subset_key not in valid_choices:
                print("Choix invalide. Veuillez entrer 0, 1, ou ALL.")

        # --- DÉBUT DU CHARGEMENT ET PRÉ-TRAITEMENT APRÈS LE PREMIER CHOIX ---

        pipeline._initialiser_et_charger_donnees()
        pipeline._charger_module_pflz()

        datasets = pipeline._preparer_subsets()
        trainer = ModelTrainer(pipeline.config)

        if chosen_subset_key == '0':
            X_data_full, y_data, subset_name = datasets['A0']
        elif chosen_subset_key == '1':
            X_data_full, y_data, subset_name = datasets['A1']
        else: # ALL
            X_data_full, y_data, subset_name = datasets['ALL']

        # --- DEUXIÈME INTERACTION UTILISATEUR (APRÈS LE CHARGEMENT DES DONNÉES) ---

        user_specific_patient_inputs = pipeline._collect_hypothetical_patient_inputs(
            selected_features=pipeline.features_selectionnees,
            X_train_for_means=pipeline.data_manager.X_train_raw
        )

        # --- CONTINUATION DU TRAITEMENT APRÈS LA DEUXIÈME QUESTION ---

        print(f"\n🔧 Processing started for {subset_name} with the user-specified characteristics...")

        print("\n" + "="*50)
        print("TREE-BASED MODEL TRAINING AND EVALUATION")
        print("="*50)

        results_train_eval_data = trainer.train_and_evaluate(
            X_data_full, y_data, subset_name, pipeline.features_selectionnees
        )

        chosen_pflz_cp_instance = results_train_eval_data['pflz_cp']
        chosen_cluster_explanations = results_train_eval_data['explanations']

        recalculated_features_interp, _, _ = InterpretabilityCalculator.calculate_interpretability_features(
            chosen_pflz_cp_instance,
            X_data_full.values,
            y_data
        )

        all_analysis_clusters = {}
        all_analysis_clusters[subset_name] = RiskClusterAnalyzer.analyze_risk_clusters(
            X=X_data_full,
            centroids=chosen_pflz_cp_instance.cluster_centers_,
            predictions=results_train_eval_data['predictions'],
            subset_name=subset_name,
            selected_features=pipeline.features_selectionnees,
            pflz_cp=chosen_pflz_cp_instance,
            cluster_explanations=chosen_cluster_explanations,
            raw_predictions=results_train_eval_data['raw_predictions'],
            y_true=y_data,
            normalized_interp_scores=recalculated_features_interp[:, 0]
        )

        final_results_summary = {subset_name: results_train_eval_data}
        pipeline._afficher_resume_final(final_results_summary, all_analysis_clusters)

        if user_specific_patient_inputs:
            print("\n" + "="*70)
            print("DETAILED ANALYSIS OF THE HYPOTHETICAL PATIENT WITH TRAINED MODELS")
            print("="*70)
            pipeline._predict_and_explain_single_instance(
                user_input_values_collected=user_specific_patient_inputs,
                pflz_cp_instance=chosen_pflz_cp_instance,
                cluster_explanations=chosen_cluster_explanations,
                selected_features=pipeline.features_selectionnees,
                X_train_for_means=pipeline.data_manager.X_train_raw
            )
        else:
            print("\n(No characteristics were specified for the hypothetical patient in the second question. Skipping detailed analysis.)")


        logger.info("✅ Pipeline executed with success.")

    except Exception as e:
        logger.error(f"💥 Fatal error: {e}")
        traceback.print_exc()
        sys.exit(1)


CHOIX DU GROUPE DE TRAITEMENT ANTIBIOTIQUE
Please choose the antibiotic treatment group for analysis:
 - '0' for patients on multiple antibiotics
 - '1' for patients on a single antibiotic
 - 'ALL' for all patients
Please choose the antibiotic treatment group for analysis (0, 1, or ALL) : 0
MOUNTING DRIVE
Mounted at /content/drive
✓ Drive mounted successfully

LOCATING DATA FOLDER
✓ modeling_data2 folder found: /content/drive/MyDrive/PFL/modeling_data2

LOADING DATA

✅ Data loaded successfully!

SYSTEM CONFIGURATION
🏥 FIXED CLINICAL THRESHOLDS:
    • LOW RISK: 10.0%
    • MEDIUM RISK: 15.0%
    • HIGH RISK: 25.0%

🔧 Functions:
    • ✅ Adaptive thresholds for coverage (60th percentile)
    • ✅ Normalization of interpretability scores (0-1)
    • ✅ Post-training calibration (isotonic)
    • ✅ Alert system for inconsistencies (threshold: 15%)
    • ✅ Increased weighting of interpretability features (boosted to 15)
    • ✅ Fixed clinical thresholds for comparability
    • ✅ Detailed loggi